<a href="https://colab.research.google.com/github/silviamoya/03MIAR-Algoritmos-Optimizacion/blob/main/SEMINARIO/Trabajo_Practico_Grupo36.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Algoritmos de optimización - Seminario<br>
**Nombre y Apellidos:** Silvia Moya Botella<br>
**Grupo:** 36. Paula Mejías Hernández y Silvia Moya Botella<br>
**Url:** https://github.com/silviamoya/03MIAR-Algoritmos-Optimizacion/blob/main/SEMINARIO/Trabajo_Practico_Grupo36.ipynb <br>
**Problema:**
> **1. Sesiones de doblaje**<br>
>2. Organizar los horarios de partidos de La Liga<br>
>3. Combinar cifras y operaciones

**Descripción del problema:**<br>

Se precisa coordinar el doblaje de una película. Los actores del doblaje deben coincidir en las tomas en las que sus personajes aparecen juntos en las diferentes tomas. Los actores de doblaje cobran todos la misma cantidad por cada día que deben desplazarse hasta el estudio de grabación independientemente del número de tomas que se graben. No es posible grabar más de 6 tomas por día.<BR>
El objetivo es planificar las sesiones por día de manera que el gasto por los servicios de los actores de doblaje sea el menor posible. Los datos son:
- Número de actores: 10
- Número de tomas : 30  
- Actores/Tomas : https://bit.ly/36D8IuK
>- 1 indica que el actor participa en la toma
>- 0 en caso contrario


---





                                        

## **Solución del problema**:
Distribuimos nuestra respuesta en tres etapas:

1. **Carga y preparación de los datos.**
2. **Backtracking con poda:**
- una primera versión sencilla;
- una segunda versión con cálculo incremental.
3. **Greedy con búsqueda local** como solución práctica.


### 1. Carga y Preparación de Datos

In [ ]:
#Librerías necesarias para la lectura de datos y los cálculos matemáticos
import pandas as pd
import math
import time



In [ ]:
sheet_id = "1Ipn6IrbQP4ax8zOnivdBIw2lN0JISkJG4fXndYd27U0"
url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv"

# Cargamos el CSV proporcionado utilizando la segunda fila como cabecera
df = pd.read_csv(url, header=1)

# Eliminamos las filas y columnas completamente vacías
df = df.dropna(axis=1, how="all")

# Mostramos las 5 primeras filas
df.head()

,Toma,1,2,3,4,5,6,7,8,9,10,Total
0,1,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,5.0
1,2,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,3.0
2,3,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,3.0
3,4,1.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,4.0
4,5,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,3.0


Se trata de un tipo de problema de planificación de tareas con restricciones y minimización de coste.

Restricciones:

*   Cada toma debe asignarse a un día
*   En cada día se pueden grabar 6 tomas como máximo
*   Si una toma usa un actor, entonces ese actor debe estar convocado ese día.

Para intentar solucionar este problema vamos a utlizar las siguientes técnicas:

**Backtracking con poda ->** Vamos asignando tomas a días, comprobamos restricciones ( máximo 6 tomas al día), si una asignación ya es peor que la mejor encontrada, la descarta.



### 2. Backtracking con poda

In [13]:
# Las filas representan las tomas y las columnas representan los actores.
# Eliminamos las columnas "Total" y "Toma" ya que no intervienen en el algoritmo.

tomas_actores = df.iloc[1:-1, 1:-1].values.tolist()

# Número máximo de tomas que pueden grabarse en un mismo día.
Max_tomas_dias = 6

# Dimensiones del problema.
num_tomas = len(tomas_actores)
num_actores = len(tomas_actores[0])

print(f"Número de tomas: {num_tomas}")
print(f"Número de actores: {num_actores}")

Número de tomas: 30
Número de actores: 10


In [11]:
#Empezamos a desarrollar

mejor_coste = math.inf #al principio suponemos que la mejor solución vale infinito
mejor_plan = None #inicialmente vale none porque no hemos encontrado ninguna solución completa

#Definimos la función de coste: ¿cuanto cuesta esta planificación?
def coste_plan(dias):
  coste = 0
  for dia in dias:
    actores_del_dia = set()
    for toma in dia:
      for actor, aparece in enumerate(tomas_actores[toma]):
       if aparece == 1:
        actores_del_dia.add(actor)
    coste += len(actores_del_dia)

  return coste

def backtracking(toma_actual, dias):
  global mejor_coste, mejor_plan
  if toma_actual == num_tomas:
    coste = coste_plan(dias) #si ya hemos colocado todas las tomas, cual es el coste?

    if coste < mejor_coste:
      mejor_coste = coste
      mejor_plan = [dia.copy() for dia in dias] #si es mejor copiamos la mejor solución día por día

    return

  #Poda
  coste_actual = coste_plan(dias)
  if coste_actual >= mejor_coste:
    return

  for dia in dias:
    if len(dia) < Max_tomas_dias:
      dia.append(toma_actual)
      backtracking(toma_actual + 1, dias)
      dia.pop()

  dias.append([toma_actual])
  backtracking(toma_actual +1, dias)
  dias.pop()

backtracking(0,[])

print("Mejor coste:", mejor_coste)
print("Mejor planificacion:")

for i,dia in enumerate(mejor_plan, start=1):
  print(f"Día {i}: tomas {dia}")

KeyboardInterrupt: 

Usando la técnica de backtracking con poda se exploran todas las soluciones posibles. Aunque este método garantiza encontrar la solución óptima, su coste computacional es exponencial, por lo que el tiempo de ejecución aumenta considerablemente cuando el número de tomas es elevado. En nuestro caso, con las 30 tomas del problema, la ejecución no finaliza en un tiempo razonable.

Para mejorar la eficiencia, en lugar de recalcular el coste completo de la planificación en cada llamada recursiva, se almacenarán los actores asignados a cada día mediante conjuntos (sets). De este modo, al añadir una nueva toma solo será necesario actualizar la información del día afectado, reduciendo el número de operaciones realizadas y mejorando el rendimiento del algoritmo.

In [ ]:
actores_por_toma = []

#Agregamos los actores por toma en la lista
for toma in tomas_actores:
    actores = set()

    for actor, aparece in enumerate(toma):
        if aparece == 1:
            actores.add(actor)

    actores_por_toma.append(actores)

# Número mínimo de días necesarios
num_dias = math.ceil(num_tomas / Max_tomas_dias)

#Empezamos con un valor de infinito como el mejor coste
mejor_coste = math.inf
mejor_plan = None

#Definimos la función de backtracking con poda
def backtracking(toma_actual, dias, actores_por_dia, coste_actual):
    global mejor_coste, mejor_plan

    #Cuando ya hayamos colocado todas las tomas miramos si es la mejor planificación encontrada hasta ahora y la guardamos en ese caso
    if toma_actual == num_tomas:
        if coste_actual < mejor_coste:
            mejor_coste = coste_actual
            mejor_plan = [dia.copy() for dia in dias]
        return

    #Si no, hacemos poda
    if coste_actual >= mejor_coste:
        return

    #Aplicamos la mejora. Guardamos el conjunto de actores que participan en la toma actual.
    actores_toma = actores_por_toma[toma_actual]


    # Meter la toma en un día existente analizando los actores ya existentes en ese dia
    for i in range(len(dias)):

        if len(dias[i]) < Max_tomas_dias:

            actores_antes = actores_por_dia[i].copy()

            actores_nuevos = actores_toma - actores_por_dia[i]
            incremento_coste = len(actores_nuevos)

            dias[i].append(toma_actual)
            actores_por_dia[i].update(actores_toma)

            backtracking(
                toma_actual + 1,
                dias,
                actores_por_dia,
                coste_actual + incremento_coste
            )

            # Deshacer cambios
            dias[i].pop()
            actores_por_dia[i] = actores_antes

    # Si todos los días que existen ya tienen 6 tomas, creamos un día nuevo
    dias.append([toma_actual])
    actores_por_dia.append(actores_toma.copy())

    backtracking(
        toma_actual + 1,
        dias,
        actores_por_dia,
        coste_actual + len(actores_toma)
    )

    # Deshacer día nuevo
    dias.pop()
    actores_por_dia.pop()

inicio = time.perf_counter()

backtracking(0, [], [], 0)

fin = time.perf_counter()

print("Mejor coste:", mejor_coste)
print("Mejor planificación:")

for i, dia in enumerate(mejor_plan, start=1):
    print(f"Día {i}: tomas {[toma + 1 for toma in dia]}")

print(f"Tiempo de ejecución: {fin - inicio:.4f} segundos")

Mejor coste: 26
Mejor planificación:
Día 1: tomas [1, 4, 5, 6, 10, 12]
Día 2: tomas [2, 3, 7, 14, 20, 28]
Día 3: tomas [8, 15, 24, 27, 29, 30]
Día 4: tomas [9, 11, 19, 21, 25, 26]
Día 5: tomas [13, 16, 17, 18, 22, 23]
Tiempo de ejecución: 974.8487 segundos


In [ ]:
print(mejor_plan)
print(mejor_coste)
print(coste_plan(mejor_plan))

[[0, 3, 4, 5, 9, 11], [1, 2, 6, 13, 19, 27], [7, 14, 23, 26, 28, 29], [8, 10, 18, 20, 24, 25], [12, 15, 16, 17, 21, 22]]
26
26


Para comparar, vamos a utilizar una técnica que no garantice encontrar el óptimo pero que el coste operacional sea más bajo.

Utilizaremos una mezcla de dos técnicas. Iniciaremos con Greedy que irá colocando cada toma en el día donde añada menos actores nuevos y si no cabe en ningún día, creará un día nuevo.
Luego usaremos búsqueda local para intentar mejorar la solución moviendo tomas entre días, incluso intercambiar dos tomas entre días. Si el coste baja, aceptará el cambio.


### 3. Greedy con búsqueda local como solución práctica.


In [14]:
#Como antes, agregamos los actores por toma en la lista
actores_por_toma = []

for toma in tomas_actores:
    actores = set()

    for actor, aparece in enumerate(toma):
        if aparece == 1:
            actores.add(actor)

    actores_por_toma.append(actores)

#Calculamos el coste agregando los actores por toma por día.
def calcular_coste(plan):
    coste = 0

    for dia in plan:
        actores_dia = set()

        for toma in dia:
            actores_dia.update(actores_por_toma[toma])

        coste += len(actores_dia)

    return coste

#Definimos el greedy inicial
def construir_solucion_greedy():
    plan = []
    actores_por_dia = []

    # Ordenamos primero las tomas con más actores
    def numero_actores(toma):
      return len(actores_por_toma[toma])

    orden_tomas = list(range(num_tomas))
    orden_tomas.sort(key=numero_actores, reverse=True)

    for toma in orden_tomas:
        mejor_dia = None
        menor_incremento = float("inf")

        # Intentar meter la toma en un día existente
        for i in range(len(plan)):
            if len(plan[i]) < Max_tomas_dias:
                actores_nuevos = actores_por_toma[toma] - actores_por_dia[i]
                incremento = len(actores_nuevos)

                if incremento < menor_incremento:
                    menor_incremento = incremento
                    mejor_dia = i

        # Si cabe en algún día, la metemos en el mejor
        if mejor_dia is not None:
            plan[mejor_dia].append(toma)
            actores_por_dia[mejor_dia].update(actores_por_toma[toma])

        # Si no cabe, creamos un día nuevo
        else:
            plan.append([toma])
            actores_por_dia.append(actores_por_toma[toma].copy())

    return plan

#Ahora aplicaremos la busqueda local para mejorar el plan
def busqueda_local(plan):
    mejora = True

    while mejora:
        mejora = False
        coste_actual = calcular_coste(plan)

        # Probar mover una toma de un día a otro
        for origen in range(len(plan)):
            for toma in plan[origen].copy():
                for destino in range(len(plan)):

                    if origen == destino:
                        continue

                    if len(plan[destino]) >= Max_tomas_dias:
                        continue

                    plan[origen].remove(toma)
                    plan[destino].append(toma)

                    nuevo_plan = [dia for dia in plan if len(dia) > 0]
                    nuevo_coste = calcular_coste(nuevo_plan)

                    if nuevo_coste < coste_actual:
                        plan = nuevo_plan
                        mejora = True
                        break

                    plan[destino].remove(toma)
                    plan[origen].append(toma)

                if mejora:
                    break

            if mejora:
                break

        if mejora:
            continue

        # Probar intercambiar dos tomas
        for dia1 in range(len(plan)):
            for dia2 in range(dia1 + 1, len(plan)):

                for toma1 in plan[dia1].copy():
                    for toma2 in plan[dia2].copy():

                        plan[dia1].remove(toma1)
                        plan[dia2].remove(toma2)

                        plan[dia1].append(toma2)
                        plan[dia2].append(toma1)

                        nuevo_coste = calcular_coste(plan)

                        if nuevo_coste < coste_actual:
                            mejora = True
                            break

                        plan[dia1].remove(toma2)
                        plan[dia2].remove(toma1)

                        plan[dia1].append(toma1)
                        plan[dia2].append(toma2)

                    if mejora:
                        break

                if mejora:
                    break

            if mejora:
                break

    return plan

#Ejecutamos las funciones definidas
plan_inicial = construir_solucion_greedy()
coste_inicial = calcular_coste(plan_inicial)

plan_final = busqueda_local(plan_inicial)
coste_final = calcular_coste(plan_final)

print("Coste inicial greedy:", coste_inicial)
print("Coste final tras búsqueda local:", coste_final)

print("\nPlanificación final:")
for i, dia in enumerate(plan_final, start=1):
  print(f"Día {i}: tomas {[toma + 1 for toma in dia]}")

Coste inicial greedy: 37
Coste final tras búsqueda local: 28

Planificación final:
Día 1: tomas [12, 25, 5, 6, 9, 28]
Día 2: tomas [3, 10, 2, 19, 14, 4]
Día 3: tomas [21, 24, 15, 8, 11, 7]
Día 4: tomas [16, 17, 18, 20, 23, 13]
Día 5: tomas [26, 27, 29, 30, 22, 1]


In [20]:
print("Plan inicial:", plan_inicial)
print("Coste inicial:", coste_inicial)
print("Plan final:", plan_final)
print("Coste final:", coste_final)

Plan inicial: [[11, 24, 4, 5, 8, 27], [2, 9, 1, 18, 13, 3], [20, 23, 14, 7, 10, 6], [15, 16, 17, 19, 22, 12], [25, 26, 28, 29, 21, 0]]
Coste inicial: 37
Plan final: [[11, 24, 4, 5, 8, 27], [2, 9, 1, 18, 13, 3], [20, 23, 14, 7, 10, 6], [15, 16, 17, 19, 22, 12], [25, 26, 28, 29, 21, 0]]
Coste final: 28


(*)¿Cuantas posibilidades hay sin tener en cuenta las restricciones?<br>



¿Cuantas posibilidades hay teniendo en cuenta todas las restricciones.




Respuesta

Modelo para el espacio de soluciones<br>
(*) ¿Cual es la estructura de datos que mejor se adapta al problema? Argumentalo.(Es posible que hayas elegido una al principio y veas la necesidad de cambiar, arguentalo)


Respuesta

Según el modelo para el espacio de soluciones<br>
(*)¿Cual es la función objetivo?

(*)¿Es un problema de maximización o minimización?

Respuesta

Diseña un algoritmo para resolver el problema por fuerza bruta

Respuesta

Calcula la complejidad del algoritmo por fuerza bruta

Respuesta

(*)Diseña un algoritmo que mejore la complejidad del algortimo por fuerza bruta. Argumenta porque crees que mejora el algoritmo por fuerza bruta

Respuesta

(*)Calcula la complejidad del algoritmo

Respuesta

Según el problema (y tenga sentido), diseña un juego de datos de entrada aleatorios

Respuesta

Aplica el algoritmo al juego de datos generado

Respuesta

Enumera las referencias que has utilizado(si ha sido necesario) para llevar a cabo el trabajo

Respuesta

Describe brevemente las lineas de como crees que es posible avanzar en el estudio del problema. Ten en cuenta incluso posibles variaciones del problema y/o variaciones al alza del tamaño

Respuesta